# 01 — Star Ratings Pipeline

## Purpose
Combine all 12 quarterly Star Ratings snapshots (May 2023 → Feb 2026) into a single clean file,
then enrich with SA3 geographic codes by joining to the service list.

## Input
- `data/raw/star_ratings/` — 12 XLSX files, one per quarterly release

## Output
- `data/clean/stars_timeline.csv` — facility × snapshot, with quality scores and SA3 codes

## Used in
- **Chapter 1** (trend line: did the Oct 2023 staffing mandate improve quality?)
- **Chapter 4** (SA3-level quality score for the master join)

## Key context
- Star Ratings launched **Dec 2022** — all data here is post-launch
- **Oct 2023** staffing mandate: 400 care mins/day, 40 RN mins/day — major turning point
- Quality score = mean of 4 sub-dimensions: Residents' Experience, Staffing, Compliance, Quality Measures
- MMM Code: MM1=Metro → MM7=Very Remote (7 remoteness classes)

In [13]:
import pandas as pd
import os

RAW_STARS   = '../../data/raw/star_ratings'
RAW_SERVICE = '../../data/raw/service_list'
OUT         = '../../data/clean/stars_timeline.csv'

# Month name → number mapping for parsing snapshot dates
MONTH_MAP = {
    'January':1, 'February':2, 'March':3, 'April':4,
    'May':5, 'June':6, 'July':7, 'August':8,
    'September':9, 'October':10, 'November':11, 'December':12
}

In [14]:
# =============================================================================
# STEP 1: Load and combine all 12 star ratings files
# =============================================================================
# Each file covers one quarterly snapshot. Column names are mostly consistent
# across files, with minor differences (e.g. 'Service Suburb' added later).
# We stack them all into one DataFrame and track the source file for debugging.

frames = []
for fname in sorted(os.listdir(RAW_STARS)):
    if not fname.endswith('.xlsx'):
        continue
    path  = f'{RAW_STARS}/{fname}'
    xl    = pd.ExcelFile(path)
    # The data sheet is always named with 'Star' — skip Notes/Contents sheets
    sheet = [s for s in xl.sheet_names if 'Star' in s or 'star' in s][0]
    df    = pd.read_excel(path, sheet_name=sheet)
    df['source_file'] = fname   # track origin for debugging
    frames.append(df)

stars = pd.concat(frames, ignore_index=True)
print(f'Combined shape: {stars.shape}')
print(f'Snapshots found: {stars["Reporting Period"].unique()}')

Combined shape: (31290, 16)
Snapshots found: <ArrowStringArray>
[  'August 2023',   'August 2025', 'December 2023', 'February 2024',
  'January 2025', 'February 2026',     'July 2024',      'May 2023',
      'May 2024',      'May 2025', 'November 2024',  'October 2025']
Length: 12, dtype: str


In [15]:
# =============================================================================
# STEP 2: Standardise column names and fix data issues
# =============================================================================

stars = stars.rename(columns={
    'Reporting Period'             : 'snapshot',
    'State/Territory'              : 'state',
    'Aged Care Planning Region'    : 'acpr_name',
    'MMM Code'                     : 'mmm_code',
    'MMM Region'                   : 'mmm_region',
    'Overall Star Rating'          : 'overall_rating',
    "Residents' Experience rating": 'residents_exp',
    'Staffing rating'              : 'staffing',
    'Compliance rating'            : 'compliance',
    'Quality Measures rating'      : 'quality_measures',
})

# Known data issue: the February 2025 file has 'January 2025' in Reporting Period.
# We detect this by cross-referencing the file name and correct it.
stars['snapshot'] = stars.apply(
    lambda r: 'February 2025'
    if r['snapshot'] == 'January 2025' and 'february-2025' in str(r['source_file'])
    else r['snapshot'],
    axis=1
)

# Convert rating columns to numeric (some cells contain suppression markers like 'np')
rating_cols = ['residents_exp', 'staffing', 'compliance', 'quality_measures', 'overall_rating']
for c in rating_cols:
    stars[c] = pd.to_numeric(stars[c], errors='coerce')

# Quality score = equal-weighted mean of 4 sub-dimensions (1–5 scale)
# Note: overall_rating is published separately and may differ slightly from this mean
stars['quality_score'] = stars[['residents_exp','staffing','compliance','quality_measures']].mean(axis=1)

# Parse snapshot string → proper datetime for time-series ordering
def parse_snapshot(s):
    parts = s.split()
    return pd.Timestamp(year=int(parts[1]), month=MONTH_MAP[parts[0]], day=1)

stars['snapshot_date'] = stars['snapshot'].apply(parse_snapshot)

print('Snapshots after fix:')
print(stars[['snapshot','snapshot_date']].drop_duplicates().sort_values('snapshot_date').to_string(index=False))

Snapshots after fix:
     snapshot snapshot_date
     May 2023    2023-05-01
  August 2023    2023-08-01
December 2023    2023-12-01
February 2024    2024-02-01
     May 2024    2024-05-01
    July 2024    2024-07-01
November 2024    2024-11-01
February 2025    2025-02-01
     May 2025    2025-05-01
  August 2025    2025-08-01
 October 2025    2025-10-01
February 2026    2026-02-01


In [16]:
# =============================================================================
# STEP 3: Join SA3 codes — multi-year service list lookup (2025 → 2024 → 2023)
# =============================================================================
# Star ratings files don't include SA3 codes — we get them from the service list.
#
# Problem with 2025-only lookup: facilities that closed before 2025 won't appear
# in the 2025 service list, leaving ~7% unmatched. Older snapshots (2023, 2024)
# naturally reference facilities that were still operating then.
#
# Strategy:
#   1. Build SA3 lookups from all three years (2023, 2024, 2025)
#   2. Join on 2025 first (most current SA3 assignment)
#   3. Fill remaining NaN from 2024, then 2023
#
# This is correct because SA3 boundaries don't change between these years —
# we're just trying to find which SA3 the facility belongs to, regardless of
# whether it's still operating.

def build_sa3_lookup(year):
    fname = next(f for f in sorted(os.listdir(RAW_SERVICE))
                 if str(year) in f and not f.startswith('~'))
    probe = pd.read_excel(f'{RAW_SERVICE}/{fname}', header=None, nrows=6)
    hdr   = next(i for i, r in probe.iterrows() if 'Service Name' in r.values)
    sl    = pd.read_excel(f'{RAW_SERVICE}/{fname}', header=hdr)
    sa3_col      = next(c for c in sl.columns if 'SA3 Code' in str(c))
    sa3_name_col = next(c for c in sl.columns if 'SA3 Name' in str(c))
    return (
        sl[['Service Name', sa3_col, sa3_name_col]]
        .drop_duplicates('Service Name')
        .rename(columns={sa3_col: 'sa3_code', sa3_name_col: 'sa3_name'})
    )

lookups = {yr: build_sa3_lookup(yr) for yr in [2025, 2024, 2023]}
for yr, lk in lookups.items():
    print(f'Service list {yr}: {len(lk):,} unique facility names')

# Join 2025 first
stars = stars.merge(lookups[2025], on='Service Name', how='left')
print(f'\nAfter 2025 join: {stars["sa3_code"].notna().mean()*100:.1f}% matched')

# Fill from 2024
mask = stars['sa3_code'].isna()
fill24 = stars.loc[mask, ['Service Name']].merge(lookups[2024], on='Service Name', how='left')
stars.loc[mask, 'sa3_code'] = fill24['sa3_code'].values
stars.loc[mask, 'sa3_name'] = fill24['sa3_name'].values
newly_filled_24 = stars['sa3_code'].notna().sum() - (stars.shape[0] - mask.sum())
print(f'After 2024 fill: {stars["sa3_code"].notna().mean()*100:.1f}% matched  (+{mask.sum() - stars["sa3_code"].isna().sum()} rows)')

# Fill from 2023
mask = stars['sa3_code'].isna()
fill23 = stars.loc[mask, ['Service Name']].merge(lookups[2023], on='Service Name', how='left')
stars.loc[mask, 'sa3_code'] = fill23['sa3_code'].values
stars.loc[mask, 'sa3_name'] = fill23['sa3_name'].values
print(f'After 2023 fill: {stars["sa3_code"].notna().mean()*100:.1f}% matched  (+{mask.sum() - stars["sa3_code"].isna().sum()} rows)')

final_unmatched = stars['sa3_code'].isna().sum()
print(f'\nFinal unmatched: {final_unmatched} rows — these are facilities not found in any 2023–2025 service list')
if final_unmatched > 0:
    print('Sample (likely pre-2023 closures or data entry inconsistencies):')
    print(stars[stars['sa3_code'].isna()]['Service Name'].value_counts().head(10).to_string())

Service list 2025: 5,238 unique facility names
Service list 2024: 5,275 unique facility names
Service list 2023: 5,364 unique facility names

After 2025 join: 93.2% matched
After 2024 fill: 97.7% matched  (+1390 rows)
After 2023 fill: 99.6% matched  (+595 rows)

Final unmatched: 134 rows — these are facilities not found in any 2023–2025 service list
Sample (likely pre-2023 closures or data entry inconsistencies):
Service Name
Ainsley Nursing Home      3
Aeralife Pennant Hills    3
Melrose Lodge             3
Arcare Aranda             3
Mountain View Lodge       3
Jacaranda Grove           3
Aeralife Illawong         3
Aeralife Botany           3
BaptistCare Minnamurra    3
Aeralife Kingswood        3


In [17]:
# =============================================================================
# STEP 4: Sanity check — quality before vs after mandate by remoteness
# =============================================================================

before = stars[stars['snapshot'] == 'August 2023'].groupby('mmm_code')['quality_score'].mean()
after  = stars[stars['snapshot'] == 'February 2024'].groupby('mmm_code')['quality_score'].mean()

check = pd.DataFrame({'Aug_2023 (pre-mandate)': before, 'Feb_2024 (post-mandate)': after})
check['change'] = check['Feb_2024 (post-mandate)'] - check['Aug_2023 (pre-mandate)']
check['pct_change'] = (check['change'] / check['Aug_2023 (pre-mandate)'] * 100).round(1)
print('Quality change by remoteness (mandate Oct 2023):')
print(check.round(3))
print()
print('>> INSIGHT: If MM6/MM7 show bigger improvements, the mandate may have forced')
print('   underperforming remote facilities to lift their game. But check if facility')
print('   COUNT also changed — some may have closed rather than complied.')

Quality change by remoteness (mandate Oct 2023):
          Aug_2023 (pre-mandate)  Feb_2024 (post-mandate)  change  pct_change
mmm_code                                                                     
MM1                        3.401                    3.499   0.098         2.9
MM2                        3.403                    3.487   0.084         2.5
MM3                        3.361                    3.393   0.032         1.0
MM4                        3.455                    3.592   0.137         4.0
MM5                        3.695                    3.801   0.106         2.9
MM6                        3.643                    3.840   0.197         5.4
MM7                        3.583                    3.906   0.323         9.0

>> INSIGHT: If MM6/MM7 show bigger improvements, the mandate may have forced
   underperforming remote facilities to lift their game. But check if facility
   COUNT also changed — some may have closed rather than complied.


In [18]:
# =============================================================================
# STEP 5: Save
# =============================================================================

keep_cols = [
    'snapshot', 'snapshot_date',
    'Service Name', 'Provider Name',
    'state', 'acpr_name', 'mmm_code', 'mmm_region', 'Size', 'Purpose',
    'overall_rating', 'residents_exp', 'staffing', 'compliance', 'quality_measures',
    'quality_score',
    'sa3_code', 'sa3_name'
]
keep_cols = [c for c in keep_cols if c in stars.columns]

out_df = stars[keep_cols].copy()
out_df.to_csv(OUT, index=False)

print(f'Saved: {out_df.shape[0]:,} rows × {out_df.shape[1]} columns → {OUT}')
print(f'Snapshots: {out_df["snapshot"].nunique()}')
print(f'Unique facilities: {out_df["Service Name"].nunique():,}')
print(f'SA3 regions covered: {out_df["sa3_code"].nunique()}')

Saved: 31,290 rows × 18 columns → ../../data/clean/stars_timeline.csv
Snapshots: 12
Unique facilities: 3,011
SA3 regions covered: 323
